# 02 · Compute Times

對每個分析單位（**里** 或 **網格 cell**）的質心，計算到最近的台南市立圖書館的時間 + 距離。

三個獨立 axis：

- **`UNIT`**：`village`（650 里，依行政邊界）或 `grid`（網格 cells，更平滑連續）
- **`METHOD`**：`haversine_30kmh`（直線距離 ÷ 30 km/h，粗略）或 `osrm`（真實道路路徑）
- **`PROFILE`**（OSRM only）：`driving`（公開 router.project-osrm.org，車用）或 `walking`（FOSSGIS routed-foot，~4.5 km/h）

輸出檔依 unit + profile 命名：
- `data/processed/{village|grid}_to_nearest_library_{driving|walking}.csv`
- 進度檔：`data/processed/osrm_progress_{UNIT}_{PROFILE}.csv`（每組合獨立斷點續跑）

`GRID_CELL_M` 控制網格大小（1000m 約 2400 cells、500m 約 9300 cells、200m 約 57000 cells）。


In [6]:
import sys
from pathlib import Path

import geopandas as gpd
import pandas as pd
from tqdm.notebook import tqdm

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from lib.geo import haversine_km, drive_minutes_from_km, safe_centroid_latlon
from lib.grid import generate_grid

RAW_DIR = ROOT / "data" / "raw"
PROC_DIR = ROOT / "data" / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)

VILLAGES_IN = RAW_DIR / "tainan_villages.geojson"
LIBRARIES_IN = RAW_DIR / "tainan_libraries.csv"

# ===== 設定 =====
# UNIT = "village"             # "village" 或 "grid"
UNIT = "grid"             # "village" 或 "grid"
GRID_CELL_M = 1000           # grid cell 邊長（公尺），UNIT="grid" 才有效
METHOD = "osrm"              # "haversine_30kmh" 或 "osrm"
PROFILE = "driving"          # OSRM 用：driving / walking
SPEED_KMH = 30.0             # haversine fallback 用

# 輸出檔依 unit + profile 命名（haversine 視為 driving）
_profile_key = PROFILE if METHOD == "osrm" else "driving"
OUTPUT = PROC_DIR / f"{UNIT}_to_nearest_library_{_profile_key}.csv"
print(f"UNIT={UNIT}, METHOD={METHOD}, PROFILE={_profile_key}")
print(f"OUTPUT={OUTPUT.name}")
if UNIT == "grid":
    print(f"GRID_CELL_M={GRID_CELL_M}")

UNIT=grid, METHOD=osrm, PROFILE=driving
OUTPUT=grid_to_nearest_library_driving.csv
GRID_CELL_M=1000


In [7]:
villages = gpd.read_file(VILLAGES_IN)
libraries = pd.read_csv(LIBRARIES_IN)
print(f"Villages: {len(villages)}, Libraries: {len(libraries)}")

if UNIT == "village":
    # Use village centroids as the analysis points
    centroids = villages.geometry.apply(
        lambda g: safe_centroid_latlon(g, source_crs="EPSG:4326")
    )
    villages["centroid_lat"] = [c[0] for c in centroids]
    villages["centroid_lon"] = [c[1] for c in centroids]
    units_df = villages.rename(columns={"village_id": "unit_id"})[
        ["unit_id", "village_name", "district", "centroid_lat", "centroid_lon", "geometry"]
    ]
elif UNIT == "grid":
    # Generate grid cells; cache to disk (regenerate if cell size changes)
    GRID_CACHE = PROC_DIR / f"grid_{GRID_CELL_M}m.geojson"
    if GRID_CACHE.exists():
        units_df = gpd.read_file(GRID_CACHE)
        print(f"Loaded cached grid: {len(units_df)} cells from {GRID_CACHE.name}")
    else:
        print(f"Generating grid... ({GRID_CELL_M}m cells)")
        units_df = generate_grid(villages, cell_size_m=GRID_CELL_M)
        units_df.to_file(GRID_CACHE, driver="GeoJSON")
        print(f"Saved {len(units_df)} cells to {GRID_CACHE.name}")
    units_df = units_df.rename(columns={"cell_id": "unit_id"})
else:
    raise ValueError(f"Unknown UNIT={UNIT!r}; use 'village' or 'grid'")

print(f"Total units: {len(units_df)}")
units_df.head()

Villages: 650, Libraries: 45
Loaded cached grid: 2412 cells from grid_1000m.geojson
Total units: 2412


,unit_id,centroid_lat,centroid_lon,geometry
0,grid_R000_C033,22.883237,120.351863,"POLYGON ((120.35676 22.87874, 120.35671 22.887..."
1,grid_R001_C023,22.891840,120.254356,"POLYGON ((120.25925 22.88735, 120.2592 22.8963..."
2,grid_R001_C025,22.891930,120.273848,"POLYGON ((120.27875 22.88744, 120.2787 22.8964..."
3,grid_R001_C026,22.891975,120.283595,"POLYGON ((120.28849 22.88748, 120.28844 22.896..."
4,grid_R001_C030,22.892146,120.322581,"POLYGON ((120.32748 22.88765, 120.32743 22.896..."


In [8]:
def nearest_library_haversine(lat: float, lon: float) -> dict:
    distances = libraries.apply(
        lambda r: haversine_km(lat, lon, r["lat"], r["lon"]),
        axis=1,
    )
    idx = distances.idxmin()
    return {
        "nearest_library": libraries.at[idx, "name"],
        "library_lat": libraries.at[idx, "lat"],
        "library_lon": libraries.at[idx, "lon"],
        "distance_km": float(distances.at[idx]),
    }


if METHOD == "haversine_30kmh":
    rows = []
    for _, u in tqdm(units_df.iterrows(), total=len(units_df), desc="haversine"):
        n = nearest_library_haversine(u["centroid_lat"], u["centroid_lon"])
        row = {
            "unit_id": u["unit_id"],
            "centroid_lat": u["centroid_lat"],
            "centroid_lon": u["centroid_lon"],
            **n,
            "time_min": drive_minutes_from_km(n["distance_km"], speed_kmh=SPEED_KMH),
            "method": METHOD,
        }
        # village-only extra columns
        if UNIT == "village":
            row["village_name"] = u["village_name"]
            row["district"] = u["district"]
        rows.append(row)
    result = pd.DataFrame(rows)
    result.to_csv(OUTPUT, index=False, encoding="utf-8-sig")
    print(f"✅ Saved {len(result)} rows to {OUTPUT}")
    result.head()

In [9]:
from lib.osrm import OSRMClient, OSRMError, load_progress, save_progress_row

OSRM_SERVERS = {
    "driving": ("https://router.project-osrm.org", "driving"),
    "walking": ("https://routing.openstreetmap.de/routed-foot", "walking"),
}

OSRM_PROGRESS = PROC_DIR / f"osrm_progress_{UNIT}_{PROFILE}.csv"
N_CANDIDATES = 3
OSRM_REQUEST_DELAY_S = 0.2

# Progress CSV schema (unit_id replaces village_id)
OSRM_FIELDS = (
    "unit_id", "nearest_library", "library_lat", "library_lon",
    "distance_km", "time_min", "method",
)


def compute_osrm_for_unit(client, u_lat, u_lon):
    candidates = libraries.copy()
    candidates["hav_km"] = candidates.apply(
        lambda r: haversine_km(u_lat, u_lon, r["lat"], r["lon"]), axis=1
    )
    top = candidates.nsmallest(N_CANDIDATES, "hav_km")

    best = None
    method = f"osrm_{PROFILE}"
    for _, lib in top.iterrows():
        try:
            time_min, osrm_km = client.route_summary(
                u_lat, u_lon, lib["lat"], lib["lon"]
            )
        except OSRMError:
            continue
        if best is None or time_min < best["time_min"]:
            best = {
                "nearest_library": lib["name"],
                "library_lat": float(lib["lat"]),
                "library_lon": float(lib["lon"]),
                "distance_km": osrm_km,
                "time_min": time_min,
            }

    if best is None:
        fb = nearest_library_haversine(u_lat, u_lon)
        best = {
            **fb,
            "time_min": drive_minutes_from_km(fb["distance_km"], SPEED_KMH),
        }
        method = f"osrm_{PROFILE}_failed_fallback"

    best["method"] = method
    return best


if METHOD == "osrm":
    # load_progress wants a generic "village_id" key — adapt to unit_id
    from lib.osrm import _FLOAT_FIELDS  # ok to use internal
    base_url, prof = OSRM_SERVERS[PROFILE]
    client = OSRMClient(base_url=base_url, profile=prof,
                        request_delay_s=OSRM_REQUEST_DELAY_S)
    print(f"Server: {base_url} (profile={prof})")

    # Load existing progress (keyed by unit_id in the file)
    import csv
    done = {}
    if OSRM_PROGRESS.exists():
        with OSRM_PROGRESS.open("r", encoding="utf-8", newline="") as f:
            for row in csv.DictReader(f):
                for k in row:
                    if k in _FLOAT_FIELDS and row[k] not in ("", None):
                        row[k] = float(row[k])
                done[row["unit_id"]] = row
    print(f"Resume: already done {len(done)} / {len(units_df)} units")

    todo = units_df[~units_df["unit_id"].isin(done.keys())].copy()
    print(f"To process: {len(todo)}")

    for _, u in tqdm(todo.iterrows(), total=len(todo), desc=f"OSRM/{UNIT}/{PROFILE}"):
        r = compute_osrm_for_unit(client, u["centroid_lat"], u["centroid_lon"])
        save_progress_row(
            OSRM_PROGRESS,
            fields=OSRM_FIELDS,
            unit_id=u["unit_id"],
            nearest_library=r["nearest_library"],
            library_lat=r["library_lat"],
            library_lon=r["library_lon"],
            distance_km=r["distance_km"],
            time_min=r["time_min"],
            method=r["method"],
        )

    # Reload and merge
    done = {}
    with OSRM_PROGRESS.open("r", encoding="utf-8", newline="") as f:
        for row in csv.DictReader(f):
            for k in row:
                if k in _FLOAT_FIELDS and row[k] not in ("", None):
                    row[k] = float(row[k])
            done[row["unit_id"]] = row

    rows = []
    for _, u in units_df.iterrows():
        d = done.get(u["unit_id"])
        if d is None:
            continue
        row = {
            "unit_id": u["unit_id"],
            "centroid_lat": u["centroid_lat"],
            "centroid_lon": u["centroid_lon"],
            **{k: d[k] for k in ["nearest_library", "library_lat", "library_lon", "distance_km", "time_min", "method"]},
        }
        if UNIT == "village":
            row["village_name"] = u["village_name"]
            row["district"] = u["district"]
        rows.append(row)
    result = pd.DataFrame(rows)
    result.to_csv(OUTPUT, index=False, encoding="utf-8-sig")
    print(f"✅ Saved {len(result)} rows to {OUTPUT}")
    n_fb = result["method"].str.contains("fallback").sum()
    if n_fb:
        print(f"⚠️  {n_fb} units fell back to haversine due to OSRM errors")

Server: https://router.project-osrm.org (profile=driving)
Resume: already done 137 / 2412 units
To process: 2275


OSRM/grid/driving:   0%|          | 0/2275 [00:00<?, ?it/s]

✅ Saved 2412 rows to /Users/linbangqi/draw-dis-to-lib/data/processed/grid_to_nearest_library_driving.csv


In [11]:
import numpy as np

result = pd.read_csv(OUTPUT)
print(f"{OUTPUT.name}: {len(result)} rows")
print(result["time_min"].describe())
print(); print("Distance (km):"); print(result["distance_km"].describe())
print()
print("By time bin:")
bins = [0, 5, 10, 15, 20, 30, 60, 120, np.inf]
print(pd.cut(result["time_min"], bins=bins, right=False).value_counts().sort_index())

grid_to_nearest_library_driving.csv: 2412 rows
count    2412.000000
mean       12.690811
std        11.525244
min         0.010000
25%         5.880000
50%         9.400000
75%        14.915833
max        86.095000
Name: time_min, dtype: float64

Distance (km):
count    2412.000000
mean        7.380413
std         7.471605
min         0.006700
25%         3.273975
50%         5.322550
75%         8.771525
max        55.916600
Name: distance_km, dtype: float64

By time bin:
time_min
[0.0, 5.0)       437
[5.0, 10.0)      865
[10.0, 15.0)     510
[15.0, 20.0)     258
[20.0, 30.0)     179
[30.0, 60.0)     126
[60.0, 120.0)     37
[120.0, inf)       0
Name: count, dtype: int64
